# Duration representations: fitting values and ordering time

The [basic representation tutorial](comparing_representations.ipynb) distinguishes selecting
a whole day from calculating values at matching timesteps. A distribution representation
uses two stages: **fit how often each attribute's values occur**, then **order those fitted
values in time**.

We use a full year of hourly solar irradiance and load to compare the basic rules with
distribution fitting, local versus global scope, and min/max preservation. Then we hold the
fitted values fixed and compare all five concurrency orderings. This separates three questions:
how well are the distributions reproduced, do the extremes survive, and which values occur
together?

## 1  A year reduced to eight typical days

The repository's hourly dataset has **8,760 samples**. We use `GHI` (solar irradiance) and
`Load`, grouped into 365 consecutive 24-hour periods starting at the first sample. The budget
is eight representative days: 192 values per attribute, each weighted by how many original
days it represents.

Every run uses the same Ward clustering and the same two attributes. Rescaling and
segmentation are disabled to expose what the representation itself changes. No data download
or random sampling is needed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution, MinMaxMean
from tsam.metrics import aggregation_summary, series_statistics
from tsam.plot import compare_series, path_panels

pio.renderers.default = "notebook_connected"
ATTRS = ["GHI", "Load"]
UNITS = {"GHI": "W/m²", "Load": "MW"}
data = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)[ATTRS]
N_CLUSTERS = 8
COLORS = {
    "original": "#68717c",
    "mean": "#0072B2",
    "medoid": "#D55E00",
    "maxoid": "#A83232",
    "minmax_mean": "#947000",
    "local": "#009E73",
    "local + min/max": "#8759A5",
    "global": "#AD3977",
    "global + min/max": "#756900",
    "independent": "#009E73",
    "medoid ordering": "#0072B2",
    "reference": "#AD3977",
    "consensus": "#8759A5",
    "assignment": "#947000",
    "real medoid": "#68717c",
}
print(
    f"{len(data):,} hourly samples, {len(data) // 24} periods, {N_CLUSTERS} typical days"
)
series_statistics({"original": data}).round(2)

## 2  Compare fitting with the basic rules

The same-timestep mean minimizes squared timing error within a fixed cluster. A local
distribution representation instead sorts **all member timesteps separately for each
attribute**, divides the sorted values into 24 equal-size groups, and averages each group.
These levels approximate that cluster's duration curve. Ordering the levels into a day is
a subsequent choice.

The configurations below keep clustering fixed. The plain names `distribution` and
`distribution_minmax` are shorthand for the two local configurations shown here.

In [ ]:
fit_rules = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "minmax_mean": MinMaxMean(max_columns=["Load"], min_columns=[]),
    "local": Distribution(),
    "local + min/max": Distribution(preserve_minmax=True),
    "global": Distribution(scope="global"),
    "global + min/max": Distribution(scope="global", preserve_minmax=True),
}
fit_results = {
    name: tsam.aggregate(
        data,
        n_clusters=N_CLUSTERS,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=rule),
        preserve_column_means=False,
    )
    for name, rule in fit_rules.items()
}
baseline = fit_results["mean"]
for result in fit_results.values():
    np.testing.assert_array_equal(
        result.cluster_assignments, baseline.cluster_assignments
    )

basic_comparison = ["mean", "medoid", "maxoid", "minmax_mean", "local"]
aggregation_summary({name: fit_results[name] for name in basic_comparison}).round(4)

`rmse` measures timing error; `rmse_duration` measures error after sorting. Both are weighted
RMSEs in **normalized units**, using the result's existing accuracy metrics. The two
correlation errors measure changes in the original/reconstructed Pearson and Spearman
matrices; they are Frobenius norms, not physical units. Lower is better.

The curves below show **physical values**. Grey is the original year, and colours identify
the methods consistently throughout this notebook. We plot each run's reconstructed year,
so common clusters contribute more hours than rare ones. Plotting the eight representative
days once each would give them the wrong weights.

In [ ]:
compare_series(
    {
        "original": data,
        **{
            name: fit_results[name].reconstructed
            for name in ["mean", "medoid", "local"]
        },
    },
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="A full year: averages, a real-day selection, or fitted distributions",
).show()

Here local distribution fitting reduces duration-curve error substantially compared with
mean and medoid, while the mean has lower timing error. The duration curve tells us how often
high or low values occur; sorting discards when they occur and how the two attributes are
paired. A better duration-curve score therefore answers only part of the question.

## 3  Choose the scope and whether to retain extremes

| Configuration | Distribution being fitted | Endpoint treatment |
|---|---|---|
| `Distribution()` | Each cluster separately | Average fitted levels |
| `Distribution(preserve_minmax=True)` | Each cluster separately | Pin its minimum and maximum, adjusting interior levels to retain the mean where feasible |
| `Distribution(scope="global")` | The whole dataset, using cluster occurrence counts | Average fitted levels |
| `Distribution(scope="global", preserve_minmax=True)` | The whole dataset | Pin the dataset's minimum and maximum, adjusting other levels where feasible |

Global fitting distributes levels across all representative days according to their
mean-profile ranks and occurrence counts. It targets the whole-year curve, and can change
individual clusters' means and distributions. Local fitting keeps each cluster's own mean
and approximates its own distribution.

In [ ]:
duration_names = ["local", "local + min/max", "global", "global + min/max"]
aggregation_summary({name: fit_results[name] for name in duration_names}).round(4)

In [ ]:
compare_series(
    {
        "original": data,
        **{name: fit_results[name].reconstructed for name in ["local", "global"]},
    },
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="Local and global fitting against the original year",
).show()

Global fitting has the smallest whole-year duration error here. Min/max preservation makes
a different promise: the original endpoints survive. A rare original peak must then occur
every time its representative day is repeated. Pinning peaks can therefore worsen the fit
to how frequently extreme values occurred; local min/max does so noticeably on this dataset.

Zoom into the **highest 5% of values** to see the endpoint trade-off. These are the upper
tails of the same whole-year curves, not a separately fitted subset.

In [ ]:
tail = compare_series(
    {
        "original": data,
        **{name: fit_results[name].reconstructed for name in duration_names},
    },
    mode="duration_curve",
    duration_range=(0, 5),
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="Highest 5% of values: keeping a peak can change its apparent frequency",
)
tail.show()

series_statistics(
    {
        "original": data,
        **{name: fit_results[name].reconstructed for name in duration_names},
    }
).round(2)

### What happens inside a cluster?

Both fitting scopes retain the dataset's mean here. Only local fitting also targets each
cluster's mean. Inspect the **largest cluster**, chosen by membership count rather than by
the quality of its fit. The table compares its original members with its local and global
representatives in physical units.

In [ ]:
focus = max(baseline.cluster_counts, key=baseline.cluster_counts.get)
members = np.flatnonzero(baseline.cluster_assignments == focus)
n_steps = baseline.n_timesteps_per_period
member_rows = (members[:, None] * n_steps + np.arange(n_steps)).ravel()
cluster_data = data.iloc[member_rows]
print(f"Cluster {focus} contains {len(members)} of the 365 days.")
series_statistics(
    {
        "original members": cluster_data,
        **{
            name: fit_results[name].cluster_representatives.loc[focus]
            for name in ["local", "global"]
        },
    }
).round(2)

## 4  Hold the values fixed and change their ordering

We now keep **local fitting without min/max preservation** fixed and vary only concurrency.
These options rearrange the same 24 fitted values for each attribute in each cluster.
They cannot improve or worsen that duration-curve fit, but they can change timing and pairing.

| Ordering | Source of the temporal ranks |
|---|---|
| `independent` | Each attribute's own mean profile (the default) |
| `medoid` | Each attribute's ranks in the same real cluster medoid |
| `reference` | One chosen attribute's mean-profile ranks for every attribute |
| `consensus` | Shared ranks from the first principal component of the standardized mean profiles |
| `assignment` | The shared ordering minimizing squared deviation from the cluster mean profile |

`medoid` **ordering** still constructs duration-curve values. It is different from the basic
`medoid` representation, which copies a member day without replacing its values.

In [ ]:
orderings = {
    "independent": Distribution(),
    "medoid ordering": Distribution(concurrency="medoid"),
    "reference": Distribution(concurrency="reference", reference_attribute="GHI"),
    "consensus": Distribution(concurrency="consensus"),
    "assignment": Distribution(concurrency="assignment"),
}
ordering_results = {
    name: tsam.aggregate(
        data,
        n_clusters=N_CLUSTERS,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=rule),
        preserve_column_means=False,
    )
    for name, rule in orderings.items()
}

# Check every cluster and attribute, not just the one we will draw.
for result in ordering_results.values():
    np.testing.assert_array_equal(
        result.cluster_assignments, baseline.cluster_assignments
    )
    for cid, profile in result.cluster_representatives.groupby(level=0):
        np.testing.assert_allclose(
            np.sort(profile.to_numpy(), axis=0),
            np.sort(
                fit_results["local"].cluster_representatives.loc[cid].to_numpy(), axis=0
            ),
        )

ordering_scores = aggregation_summary(ordering_results)
np.testing.assert_allclose(
    ordering_scores["rmse_duration"], ordering_scores["rmse_duration"].iloc[0]
)
ordering_scores.round(4)

The duration errors agree, but the other columns do not. On this year, medoid ordering
improves rank-correlation error compared with independent ordering while slightly worsening
Pearson correlation error. Borrowing real-day ranks is a candidate to evaluate, not a
guarantee of lower errors.

Reference, consensus, and assignment have identical correlation errors here, although their
timing errors differ. They pair the same ranked levels within each cluster; moving those
pairs together changes their timing without changing their joint distribution.

## 5  Follow the largest cluster through time

The metrics above cover the **whole year**. The next figures follow the same largest cluster
we inspected earlier. First compare hourly profiles with the real medoid: its values are
observed together, while all the duration profiles use fitted levels.

The horizontal axis is the timestep within the 24-hour period, not the original calendar
date. This view makes the ordering changes visible without sorting away chronology.

In [ ]:
real_medoid = fit_results["medoid"].cluster_representatives.loc[focus]
ordered_profiles = {
    name: result.cluster_representatives.loc[focus]
    for name, result in ordering_results.items()
}
compare_series(
    {
        "real medoid": real_medoid,
        **{
            name: ordered_profiles[name]
            for name in ["independent", "medoid ordering", "reference"]
        },
    },
    reference="real medoid",
    units=UNITS,
    colors=COLORS,
    title="One cluster: a real day and three ways to order fitted values",
).show()

Now trace the paired irradiance/load values as paths. The reference medoid is grey; each
panel highlights one profile in black. Arrows retain temporal direction, and hover labels
give `t0`–`t23`. Printed labels are suppressed because 24 labels would crowd the paths.

Shared-rank strategies tend to pair low values with low values and high with high within a
period. That can impose positive within-period correlation. Matching a duration curve does
not validate these pairs; ties also prevent exact rank-correlation guarantees.

In [ ]:
path_panels(
    {"real medoid": real_medoid, **ordered_profiles},
    "GHI",
    "Load",
    background={"real medoid": real_medoid},
    units=UNITS,
    colors=COLORS,
    path_color="#222222",
    path_label="highlighted profile",
    dash="dash",
    label_steps=False,
    title="One cluster: identical marginal values, different paths",
).show()

## 6  Choose and measure the final configuration

* Compare **mean and medoid** as baselines before adding distribution fitting.
* Choose **local** fitting when each cluster's own distribution and mean matter; compare
  **global** fitting when the whole-series duration curve is the target.
* Add **min/max preservation** when endpoint values matter, and check the resulting
  frequency fit and means. Simultaneous preservation depends on feasibility.
* Compare **concurrency orderings** using timing and joint-attribute metrics. Their equal
  duration-curve errors tell us nothing about which ordering fits the application best.

Non-default concurrency strategies require `Distribution(scope="local")` on `ClusterConfig`.
They can be combined with `preserve_minmax=True`; the fitted levels would then change for
all orderings. Concurrency cannot be set on `SegmentConfig`.

This tutorial deliberately omitted segmentation and rescaling. Both can alter the final
curves and pairings. Recheck the final configuration, including chronology relevant to storage
or ramping, rather than choosing from a duration-curve score alone.

* [Basic representations](comparing_representations.ipynb) — whole-day selection versus coordinate-wise construction.
* [Representations how-to](../how-to/representations.ipynb#choose-how-attributes-coincide) — configuration recipes and limits.
* [Representation algorithms](../explanation/how-aggregation-works/03_representation.ipynb) — calculations on a tiny dataset.